# Customer Churn Prediction
Dataset: 200 synthetic customer records. Goal: train a Decision Tree to predict which customers may leave, evaluate it, and find customers at risk.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv("customers.csv")
df.head()

## 1. Inspect the data

In [ ]:
df.info()
print(df.isna().sum())
print(df["churn"].value_counts())

## 2. Prepare the data

In [ ]:
df["monthly_charges"] = df["monthly_charges"].fillna(df["monthly_charges"].median())
df["churn"] = df["churn"].map({"Yes": 1, "No": 0})
print(df.groupby("contract")["churn"].mean().round(2))

## 3. Target and features

In [ ]:
y = df["churn"]
X = pd.get_dummies(df.drop(columns=["customer_id", "churn"]), columns=["contract"])
X.head()

## 4. Split into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 5. Train a Decision Tree

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
plt.figure(figsize=(14, 6))
plot_tree(model, feature_names=list(X.columns), class_names=["Stay", "Churn"], filled=True)
plt.show()

## 6. Predict and evaluate

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 2))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Stay", "Churn"]))

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values()
importance.plot(kind="barh", title="Feature Importance")
plt.xlabel("Importance")
plt.show()

## 7. Customers at risk

In [ ]:
at_risk = df.loc[X_test.index, ["customer_id", "tenure_months", "contract", "support_calls"]]
at_risk["churn_probability"] = model.predict_proba(X_test)[:, 1]
at_risk.sort_values("churn_probability", ascending=False).head(10)

## 8. Observations and retention insights
- **Data:** 200 customers, 82 of them churned (41%). 6 customers had no monthly charge, so we filled it with the median.
- **Contract:** month-to-month customers churn far more (about 70%) than one-year (about 15%) and two-year customers (about 8%).
- **Model:** a Decision Tree with depth 3 reached an accuracy of [YOUR ACCURACY] on the 50 test customers. It caught [N] of the [N] churners in the test set.
- **Most important features:** [WRITE THE TOP FEATURES FROM THE CHART].
- **Retention ideas:** offer month-to-month customers a discount to move to a yearly contract, follow up with new customers in their first months, and contact customers who call support many times.

## 9. Conclusion
We loaded and prepared 200 customer records, trained a Decision Tree, and evaluated it on customers it had never seen. The model and the at-risk list help the retention team decide which customers to contact first.